# Notebook 03 - Spectral Features

## Goal
Extract and compare MFCC, LFCC-like, mel, log spectrogram, and CQCC-like features.


## Agenda
- Compute mel spectrogram
- Compute MFCC
- Build LFCC-like features
- Build CQT-based cepstral proxy


## Concept and Math

MFCC = DCT(log(Mel filterbank energies)).
LFCC replaces Mel with linear frequency spacing before DCT.
Cepstral transforms compactly represent spectral envelope information.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No .flac or .wav found under ../dataset")

audio_path = audio_files[0]
print(f"Using: {audio_path}")

from scipy.fftpack import dct

wave, sr = lb.load(audio_path, sr=16000, mono=True)
mel = lb.feature.melspectrogram(y=wave, sr=sr, n_fft=512, hop_length=160, n_mels=80)
log_mel = lb.power_to_db(mel, ref=np.max)
mfcc = lb.feature.mfcc(S=log_mel, n_mfcc=20)

stft = lb.stft(wave, n_fft=512, hop_length=160, win_length=400)
log_power = np.log(np.abs(stft) ** 2 + 1e-8)
lfcc_like = dct(log_power, type=2, axis=0, norm="ortho")[:20]

cqt = np.abs(lb.cqt(wave, sr=sr, n_bins=84, bins_per_octave=12))
cqcc_like = dct(np.log(cqt + 1e-8), type=2, axis=0, norm="ortho")[:20]

print("log_mel:", log_mel.shape, "mfcc:", mfcc.shape)
print("lfcc_like:", lfcc_like.shape, "cqcc_like:", cqcc_like.shape)


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torchaudio

wave_t, sr_t = torchaudio.load(str(audio_path))
wave_t = wave_t.mean(dim=0, keepdim=True)
mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=sr_t, n_fft=512, hop_length=160, n_mels=80)
mfcc_tf = torchaudio.transforms.MFCC(sample_rate=sr_t, n_mfcc=20, melkwargs={"n_fft": 512, "hop_length": 160, "n_mels": 80})
print(mel_tf(wave_t).shape, mfcc_tf(wave_t).shape)


## Review Checklist
- When can LFCC help more than MFCC?
- What does DCT do here?
- What information is lost in cepstral compression?
